# Análisis Exploratorio de Datos (EDA)
## Modelo Predictivo de Demanda de Agua Potable - Gran Valparaíso

Este notebook realiza un análisis exploratorio completo de los datos de demanda de agua potable, incluyendo:
- Carga y visualización de datos
- Estadísticas descriptivas
- Análisis de patrones temporales
- Identificación de tendencias y estacionalidad
- Análisis de eventos especiales y feriados

## 1. Importar Librerías y Configuración

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import sys

# Agregar el directorio src al path
sys.path.append(str(Path().absolute().parent))

# Importar módulos personalizados
from src.utils import load_config, setup_logging, get_data_summary
from src.data_processing import DataProcessor

# Configuración de visualización
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configurar tamaño de figuras
plt.rcParams['figure.figsize'] = (15, 6)
plt.rcParams['figure.dpi'] = 100

print("✓ Librerías importadas correctamente")

In [ ]:
# Cargar configuración
config = load_config('../config/config.yaml')
setup_logging(config)

print("✓ Configuración cargada")
print(f"  - Directorio de datos raw: {config['data']['raw_dir']}")
print(f"  - Archivo de volumen: {config['data']['volume_file']}")
print(f"  - Archivo de calendario: {config['data']['calendar_file']}")

## 2. Cargar Datos

In [ ]:
# Crear procesador de datos
processor = DataProcessor(config)

# Cargar datos de volumen
print("Cargando datos de volumen...")
df_volume = processor.load_volume_data()

# Cargar datos de calendario
print("Cargando datos de calendario...")
df_calendar = processor.load_calendar_data()

# Mostrar información básica
print(f"\n✓ Datos cargados exitosamente")
print(f"  - Registros de volumen: {len(df_volume):,}")
print(f"  - Registros de calendario: {len(df_calendar):,}")

# 🎯 Comparación Predicción vs Realidad

Esta sección muestra la comparación entre las predicciones del modelo y los valores reales de demanda de agua.

In [ ]:
# Importar librerías para visualización de modelos
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from PIL import Image
from pathlib import Path
import numpy as np

# Mostrar las gráficas generadas
figures_path = Path("../outputs/figures")

print("📊 GRÁFICAS DE PREDICCIÓN vs REALIDAD GENERADAS:")
print("="*60)

# Buscar las gráficas más recientes
prediction_graphs = list(figures_path.glob("prediccion_vs_real_*.png"))
simple_graphs = list(figures_path.glob("comparacion_simple_*.png"))

if prediction_graphs:
    latest_prediction = max(prediction_graphs, key=lambda x: x.stat().st_mtime)
    print(f"📈 Gráfica completa: {latest_prediction.name}")
    
    # Mostrar la gráfica
    img = Image.open(latest_prediction)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Comparación Completa: Predicción vs Realidad', fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()

if simple_graphs:
    latest_simple = max(simple_graphs, key=lambda x: x.stat().st_mtime)
    print(f"📈 Gráfica simple: {latest_simple.name}")
    
    # Mostrar la gráfica simple
    img_simple = Image.open(latest_simple)
    plt.figure(figsize=(14, 8))
    plt.imshow(img_simple)
    plt.axis('off')
    plt.title('Comparación Simple: Predicción vs Comportamiento Real', fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()

print("\n✅ Gráficas mostradas exitosamente")
print("💡 Para generar nuevas gráficas, ejecuta: python quick_prediction_graph.py")

In [ ]:
# ENTRENAR MODELO RÁPIDO Y CREAR GRÁFICA EN VIVO
print("🤖 Entrenando modelo Random Forest para comparación...")

# Cargar datos procesados
data_path = Path("../data/processed")

if (data_path / "data_train.csv").exists():
    # Cargar datos
    train_df = pd.read_csv(data_path / "data_train.csv")
    test_df = pd.read_csv(data_path / "data_test.csv")
    
    print(f"📊 Datos cargados: {len(train_df)} train, {len(test_df)} test")
    
    # Crear features básicas rápidamente
    for df in [train_df, test_df]:
        df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])
        df['hour'] = df['timestamp_utc'].dt.hour
        df['day_of_week'] = df['timestamp_utc'].dt.dayofweek
        df['month'] = df['timestamp_utc'].dt.month
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # Preparar datos para el modelo
    target_col = ' Volumen_Total_m3'
    feature_cols = ['hour', 'day_of_week', 'month', 'is_weekend']
    
    # Agregar features de calendario si existen
    if 'feriado' in train_df.columns:
        feature_cols.append('feriado')
    if 'temporada_turistica_alta' in train_df.columns:
        feature_cols.append('temporada_turistica_alta')
    
    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df[target_col]
    X_test = test_df[feature_cols].fillna(0)
    y_test = test_df[target_col]
    
    # Entrenar modelo
    model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    
    # Hacer predicciones
    y_pred = model.predict(X_test)
    
    # Calcular métricas básicas
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    print(f"📊 MÉTRICAS DEL MODELO:")
    print(f"   RMSE: {rmse:,.0f} m³")
    print(f"   MAE: {mae:,.0f} m³") 
    print(f"   MAPE: {mape:.2f}%")
    print(f"   R²: {r2:.4f}")
    
    # CREAR GRÁFICA COMPARATIVA
    # Tomar muestra para visualización clara
    n_points = min(150, len(y_test))
    indices = np.linspace(0, len(y_test)-1, n_points, dtype=int)
    
    timestamps_sample = test_df['timestamp_utc'].iloc[indices]
    y_test_sample = y_test.iloc[indices] 
    y_pred_sample = y_pred[indices]
    
    # Crear la gráfica principal
    plt.figure(figsize=(15, 10))
    
    # Subplot 1: Comparación temporal
    plt.subplot(2, 2, 1)
    plt.plot(range(len(y_test_sample)), y_test_sample, 'b-o', 
             label='Valores Reales', linewidth=2.5, markersize=5, alpha=0.8)
    plt.plot(range(len(y_pred_sample)), y_pred_sample, 'r-s', 
             label='Predicciones', linewidth=2.5, markersize=5, alpha=0.8)
    plt.title('🎯 Predicción vs Realidad - Serie Temporal', fontsize=14, fontweight='bold')
    plt.xlabel('Observaciones (Orden Temporal)')
    plt.ylabel('Volumen (m³)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Subplot 2: Scatter plot
    plt.subplot(2, 2, 2)
    plt.scatter(y_test, y_pred, alpha=0.6, s=25, color='purple')
    
    # Línea de predicción perfecta
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    
    plt.title('📊 Scatter: Predicción vs Real', fontsize=14, fontweight='bold')
    plt.xlabel('Valores Reales (m³)')
    plt.ylabel('Predicciones (m³)')
    plt.grid(True, alpha=0.3)
    
    # Agregar métricas al gráfico
    textstr = f'R² = {r2:.3f}\\nRMSE = {rmse:,.0f}\\nMAE = {mae:,.0f}\\nMAPE = {mape:.1f}%'
    props = dict(boxstyle='round', facecolor='lightgreen', alpha=0.8)
    plt.text(0.05, 0.95, textstr, transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='top', bbox=props)
    
    # Subplot 3: Distribución de errores
    plt.subplot(2, 2, 3)
    errors = np.abs(y_test - y_pred)
    plt.hist(errors, bins=25, alpha=0.7, edgecolor='black', color='orange')
    plt.title('📈 Distribución de Errores', fontsize=14, fontweight='bold')
    plt.xlabel('Error Absoluto (m³)')
    plt.ylabel('Frecuencia')
    plt.grid(True, alpha=0.3)
    plt.axvline(errors.mean(), color='red', linestyle='--', 
                label=f'Error Promedio: {errors.mean():,.0f} m³')
    plt.legend()
    
    # Subplot 4: Importancia de features
    plt.subplot(2, 2, 4)
    importances = model.feature_importances_
    indices_imp = np.argsort(importances)[::-1]
    
    plt.bar(range(len(feature_cols)), importances[indices_imp], color='skyblue', alpha=0.8)
    plt.title('🔍 Importancia de Features', fontsize=14, fontweight='bold')
    plt.xlabel('Features')
    plt.ylabel('Importancia')
    plt.xticks(range(len(feature_cols)), [feature_cols[i] for i in indices_imp], rotation=45)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Análisis de precisión
    errors_pct = np.abs((y_test.values - y_pred) / y_test.values) * 100
    excellent = (errors_pct <= 10).sum()
    good = ((errors_pct > 10) & (errors_pct <= 25)).sum()
    fair = (errors_pct > 25).sum()
    
    print(f"\\n🎯 ANÁLISIS DE PRECISIÓN:")
    print(f"   ✅ Excelente (≤10% error): {excellent} ({excellent/len(errors_pct)*100:.1f}%)")
    print(f"   👍 Bueno (10-25% error): {good} ({good/len(errors_pct)*100:.1f}%)")
    print(f"   ⚠️ Regular (>25% error): {fair} ({fair/len(errors_pct)*100:.1f}%)")
    
    print(f"\\n🏆 CONCLUSIÓN:")
    if r2 > 0.7:
        print("   🎉 ¡Modelo excelente! Predicciones muy precisas.")
    elif r2 > 0.5:
        print("   👍 Modelo bueno. Predicciones aceptables.")
    else:
        print("   ⚠️ Modelo básico. Se puede mejorar con más features.")
        
else:
    print("❌ No se encontraron datos procesados.")
    print("💡 Ejecuta primero: python run_pipeline.py")